## **Environment Setup & Loading Data**

In [2]:
import pandas as pd
import datetime as dt

df = pd.read_csv('../data/online_retail_II.csv')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 136.6 MB


In [4]:
df.describe()

,Quantity,Price,Customer ID
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


## **Data Cleaning**

In [4]:
df = df.dropna(subset=['Customer ID'])

df = df[~df['Invoice'].str.contains('C')]

df = df[(df['Quantity'] > 0) & (df['Price']>0)]

df['TotalSpend'] = df['Quantity'] * df['Price']

In [5]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df.isna().sum()

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
TotalSpend     0
dtype: int64

## **RFM Feature Engineering**

In [6]:
latest_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

rfm_table = df.groupby('Customer ID').agg({
    'InvoiceDate' : lambda x: (latest_date - x.max()).days,
    'Invoice' : 'nunique',
    'TotalSpend' : 'sum'
})

rfm_table.rename(columns={
    'InvoiceDate' : 'Recency',
    'Invoice' : 'Frequency',
    'TotalSpend' : 'Monetary'
}, inplace=True)

rfm_table.head()

,Recency,Frequency,Monetary
Customer ID,,,
12346.0,326,12,77556.46
12347.0,2,8,5633.32
12348.0,75,5,2019.40
12349.0,19,4,4428.69
12350.0,310,1,334.40


## **Data Preprocessing (Normalization)**

In [7]:
import numpy as np
from sklearn.preprocessing import StandardScaler

rfm_log = np.log1p(rfm_table)

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log) 

## **K-Means Clustering**

In [8]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=42)
rfm_table['Cluster'] = kmeans.fit_predict(rfm_scaled)

In [10]:
cluster_summary = rfm_table.groupby('Cluster').agg({
    'Recency':'mean',
    'Frequency':'mean',
    'Monetary':'mean'
}).round(2)

cluster_summary

,Recency,Frequency,Monetary
Cluster,,,
0,394.66,1.41,344.06
1,25.51,19.89,11386.98
2,28.51,3.01,856.19
3,213.50,5.39,2139.52


## **Segment Profiling & Naming**

In [14]:
segment_map = {
    1: 'Champions',
    2: 'New / Promising',
    3: 'At-Risk',
    0: 'Lost'
}
rfm_table['Customer_Segment'] = rfm_table['Cluster'].map(segment_map)

rfm_table[['Recency', 'Frequency', 'Monetary', 'Customer_Segment']].head()

,Recency,Frequency,Monetary,Customer_Segment
Customer ID,,,,
12346.0,326,12,77556.46,Champions
12347.0,2,8,5633.32,Champions
12348.0,75,5,2019.40,At-Risk
12349.0,19,4,4428.69,New / Promising
12350.0,310,1,334.40,Lost


In [15]:
rfm_table.shape

(5878, 5)

## ***Saving the RFM Table for Visualization***

In [16]:
rfm_table.to_csv('./data/segmented_customers.csv')